In [1]:
import os
import h5py
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
from obspy.core import UTCDateTime as udt
matplotlib.use('agg')
import utils
import gc
from datetime import datetime,timedelta

### 分析图

In [2]:
def analyze_single_file(result_file, save_dir='./analysis_plots',rain_offset=0):
	os.makedirs(save_dir, exist_ok=True)
	print(f"Analyzing file: {result_file}")
	results = utils.load_result(os.path.basename(result_file), path=os.path.dirname(result_file)+'/')

	if not results:
		print(f"No valid data in {result_file}, skipping.")
		return

	# 提取文件名用于标题
	base_name = os.path.splitext(os.path.basename(result_file))[0]

	# 1. 柱状图：matched_count per template
	templates = list(results.keys())
	matched_counts = [results[t]['matched_count'] for t in templates]

	plt.figure(figsize=(12, 6))
	plt.bar(templates, matched_counts, color='skyblue')
	plt.xticks(rotation=45, ha='right')
	plt.xlabel('Template')
	plt.ylabel('Matched Event Count')
	plt.title(f'Matched Event Count - {base_name}')
	plt.tight_layout()
	plt.savefig(os.path.join(save_dir, f'{base_name}_matched_count.png'))
	plt.close()

	# 2. 折线图：Daily matched events
	#daily_data = {t: results[t]['matched_event_index_daily'] for t in templates if t!='template_16334774.h5'}
	daily_data = {t: results[t]['matched_event_index_daily'] for t in templates}
	max_days = max(len(d) for d in daily_data.values())
	plt.figure(figsize=(12, 6))
	for template, counts in daily_data.items():
		daily_counts = [len(day) for day in counts]  # 统计每一天有多少事件
		plt.plot(range(1, len(daily_counts)+1), daily_counts, marker='o', linestyle='-', label=template)
	plt.legend(loc='upper right', bbox_to_anchor=(1.2, 1), fontsize='small')
	plt.xlabel('Day Index')
	plt.ylabel('Matched Events per Day')
	plt.title(f'Daily Matched Events - {base_name}')
	plt.legend(loc='upper right', bbox_to_anchor=(1.2, 1), fontsize='small')
	plt.grid(True)
	plt.tight_layout()
	plt.savefig(os.path.join(save_dir, f'{base_name}_daily_matched_events.png'))
	plt.close()

	# 3. 全局匹配事件分布图
	all_indices = []
	for t in templates:
		all_indices.extend(results[t]['matched_event_index_all'])
	all_indices.sort()
	plt.figure(figsize=(12, 3))
	plt.plot(all_indices, np.ones_like(all_indices), '|', markersize=10, color='black')
	plt.yticks([])
	plt.xlabel('Global Sample Index')
	plt.title(f'Global Matched Events Distribution - {base_name}')
	plt.tight_layout()
	plt.savefig(os.path.join(save_dir, f'{base_name}_global_matched_events.png'))
	plt.close()
	# 4. 带雨折线图
	#daily_data = {t: results[t]['matched_event_index_daily'] for t in templates if t!='template_16334774.h5'}
	daily_data = {t: results[t]['matched_event_index_daily'] for t in templates}
	fig, ax1 = plt.subplots(figsize=(20, 10))

	# 左侧 y 轴：匹配事件数量
	for template, counts in daily_data.items():
		daily_counts = [len(day) for day in counts]  # 统计每一天有多少事件
		ax1.plot(range(1, len(daily_counts)+1), daily_counts, marker='o', linestyle='-', label=template)

	ax1.set_xlabel('Day Index')
	ax1.set_ylabel('Matched Events per Day')
	ax1.grid(True)

	# 右侧 y 轴：降雨量 (mm)
	ax2 = ax1.twinx()
	rain_data = [0.0, 0.0, 0.0, 8.4, 0.6000000000000001, 5.6, 0.8, 0.2, 0.2, 4.800000000000001, 1.8, 1.2000000000000002, 0.0, 0.6000000000000001, 1.4000000000000001, 3.4, 0.4, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 17.0, 7.2, 0.6000000000000001, 0.4, 0.0]
	x_axis_rain = range(rain_offset, rain_offset + len(rain_data))  # 假设从第3天开始有雨量数据

	ax2.plot(x_axis_rain, rain_data, color='red', linestyle='--', label='Rain (mm)')
	ax2.set_ylabel('Rain (mm)', color='red')

	# 合并图例
	lines_1, labels_1 = ax1.get_legend_handles_labels()
	lines_2, labels_2 = ax2.get_legend_handles_labels()
	ax1.legend(lines_1 + [lines_2[0]], labels_1 + labels_2, loc='upper right', bbox_to_anchor=(1.2, 1), fontsize='small')

	plt.title(f'Daily Matched Events & Rainfall - {base_name}')
	plt.tight_layout()
	plt.savefig(os.path.join(save_dir, f'{base_name}_daily_matched_events_with_rain.png'))
	plt.close()

def batch_analyze_results(input_dir='./logs', output_dir='./analysis_plots',station='Xs01',rain_offset = 0):
	# 查找所有匹配的 .h5 文件
	if not os.path.exists(output_dir):
		os.makedirs(output_dir)
	
	result_files = [f for f in os.listdir(input_dir) if f.startswith(station+'_matched_filter_results_threshold') and f.endswith('.h5')]

	if not result_files:
		print("No result files found.")
		return

	for filename in result_files:
		full_path = os.path.join(input_dir, filename)
		file_save_dir = os.path.join(output_dir, os.path.splitext(filename)[0])
		analyze_single_file(full_path, file_save_dir,rain_offset)

In [ ]:
#stations=['Xs01','Xs02','Xs03','Xs04','Xs05','Xs06','Xs07','Xs08',
#          'Xs09','Xs10','Xs11','Xs12','Xs13','Xs14','Xs16','Xs17',
#          'Xs18','Xs19','Xs20','Xs21','Xs22','Xs23','Xs24','Xs25',
#          'Xs26','Xs27','Xs28','Xs29'
#          ]
#offsets = [1, 1, 1, 1, 1, 0, 0, 0,
#           0, 0, 0, 0, 0, 2, 2, 2,
#           2, 2, 2, 2, 2, 2, 3, 3,
#           3, 3, 3, 3
#           ]
stations = ['Xs28']
offsets = [3]
for station, offset in zip(stations, offsets):
	batch_analyze_results(input_dir='./logs_814-918_overlap/'+station,
						output_dir='./analysis_plots/814-918_overlap/'+station,
						station= station,
						rain_offset = offset)

Analyzing file: ./logs_814-918_overlap/Xs01/Xs01_matched_filter_results_threshold0.7_weight0.0_1.0_0.0.h5
Analyzing file: ./logs_814-918_overlap/Xs01/Xs01_matched_filter_results_threshold0.8_weight0.0_1.0_0.0.h5
Analyzing file: ./logs_814-918_overlap/Xs01/Xs01_matched_filter_results_threshold0.8_weight0.0_0.0_1.0.h5
Analyzing file: ./logs_814-918_overlap/Xs01/Xs01_matched_filter_results_threshold0.7_weight0.0_0.0_1.0.h5
Analyzing file: ./logs_814-918_overlap/Xs01/Xs01_matched_filter_results_threshold0.8_weight1.0_0.0_0.0.h5
Analyzing file: ./logs_814-918_overlap/Xs01/Xs01_matched_filter_results_threshold0.7_weight1.0_0.0_0.0.h5
Analyzing file: ./logs_814-918_overlap/Xs02/Xs02_matched_filter_results_threshold0.8_weight0.0_1.0_0.0.h5
Analyzing file: ./logs_814-918_overlap/Xs02/Xs02_matched_filter_results_threshold0.7_weight0.0_1.0_0.0.h5
Analyzing file: ./logs_814-918_overlap/Xs02/Xs02_matched_filter_results_threshold0.7_weight0.0_0.0_1.0.h5
Analyzing file: ./logs_814-918_overlap/Xs02/Xs

### 匹配到的一些波形

In [ ]:
def extract_matched_waveforms_from_daily(template_event, n_best, result_dict, sac_events, data_dir='./', station_idx=0):
	"""
	根据 matched_event_index_daily 提取每天的匹配事件波形，并返回带有 station_idx 的 detections 字典。
	
	参数:
		template_events: list of str, 多个模板名称
		result_dict: dict, load_result() 返回的结果字典
		sac_events: list of str, 所有 sac 文件名
		data_dir: str, 原始数据路径
		station_idx: int, 当前绘制使用的台站索引（用于画图时定位台站）

	返回:
		detections: dict, key 为 template_idx，value 为对应模板的波形与元数据
	"""
	detection = {}
	# 加载模板以获取模板持续时间（单位：采样点）
	template = utils.load_template(template_event, path='./templates/')
	template_duration = template['waveforms'].shape[-1]  # 获取模板长度（采样点数）
	sampling_rate = 250  # 假设固定采样率，也可以从模板中读取

	waveforms = []
	origin_times = []
	daily_indices_list = result_dict.get(template_event, {}).get('matched_event_index_daily', [])
	if not daily_indices_list:
		print(f"No matched events found for template {template_event}, skipping.")
		return
	for day_idx, indices_in_day in enumerate(daily_indices_list):
		if not indices_in_day:
			continue  # 跳过没有事件的天
		sac_file = sac_events[day_idx]
		data = utils.load_data(sac_file, path=data_dir)
		wf = data['waveforms']
		wf_time = data['metadata']['date']
		del data
		gc.collect()
		for idx in indices_in_day:
			start_idx = idx
			end_idx = idx + template_duration
			slice_wf = wf[:, :, start_idx:end_idx]
			waveforms.append(slice_wf)
			del slice_wf
			# 防止卡死
			if len(waveforms) >= n_best:
				break
			origin_time = wf_time + idx / sampling_rate
			origin_times.append(origin_time)
		if len(waveforms) >= n_best:
			break
	detection = {
		'waveforms': np.array(waveforms),
		'metadata': {
			'origin_times': np.array(origin_times),
		}
	}

	return detection

In [ ]:
def plot_n_detections(detection, n_best, template,components, weight = 0, station_idx=0, save_pos = None):
	template_waveforms = template['waveforms'][station_idx, :, :]
	detection_waveforms = detection['waveforms']
	if(detection_waveforms.shape[0] < n_best):
		n_best = detection_waveforms.shape[0]
	template_duration =  len(template['waveforms'][0][0])
	# select subsets
	detection_waveforms = detection_waveforms[:n_best, :, :,:]
	OT = detection['metadata']['origin_times'][:n_best]
	duration = template['waveforms'].shape[-1]
	# start plotting
	time = np.linspace(0., template_duration, duration)
	figsize = (50, 30)
	plt.figure('detection', figsize=figsize)
	plt.suptitle('Station {}'.format(template['stations'][station_idx].decode('utf-8')))
	n_components = 1
	for c in range(n_components):
		plt.subplot(n_best + 1, n_components, 1 + c)
		plt.title(components[c])
		template_waveforms = (template_waveforms - np.mean(template_waveforms)) / np.std(template_waveforms)
		plt.plot(time, template_waveforms[weight, :], lw=0.75, color='C3', label='Template')
		plt.xlim(time.min(), time.max())
		if c == 2:
			plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left', borderaxespad=0., handlelength=0.1)
		for n in range(n_best):
			plt.subplot(n_best + 1, n_components, (1 + n)*n_components + c + 1)
			detection_waveforms[n, : ,weight, :][0] = (detection_waveforms[n, : ,weight, :][0] - np.mean(detection_waveforms[n, : ,weight, :][0])) / np.std(detection_waveforms[n, : ,weight, :][0])
			plt.plot(time, detection_waveforms[n, : ,weight, :][0], lw=0.75, color='C0',
					 label=udt(OT[n])\
							 .strftime('%Y,%m,%d -- %H:%M:%S'))
			plt.xlim(time.min(), time.max())
			if n == n_best - 1:
				plt.xlabel('Time (s)')
	plt.subplots_adjust(top=0.91,
			bottom=0.075,
			left=0.06,
			right=0.885,
			hspace=0.2,
			wspace=0.2)

	plt.savefig(save_pos)
	plt.close()

In [ ]:
# sac sort
def extract_date_safe(filename):
	try:
		date_str = filename.split('_')[0]
		return datetime.strptime(date_str, '%Y%m%d')
	except (IndexError, ValueError):
		# 如果解析失败，返回一个极大时间，排在最后
		return datetime.max
	
def extract_date_safe_ep(filename):
	try:
		# 假设文件名格式为 'YYYY.DDD.XXXX.h5'
		parts = os.path.basename(filename).split('.')
		year = int(parts[0])
		julian_day = int(parts[1])
		# 将儒略日转换为实际日期
		date = datetime(year, 1, 1) + timedelta(days=julian_day - 1)
		return date
	except (IndexError, ValueError):
		# 如果解析失败，返回极大时间，排在最后
		return datetime.max

# 加载结果
threshlolds = ['0.7', '0.8']
weights = [[0,'1.0_0.0_0.0'],[1,'0.0_1.0_0.0'],[2,'0.0_0.0_1.0']]
log_name = '814-918/'
result_dir = './logs_' + log_name
n_best = 8

tempates_dir = './templates/'
h5_file = './h5/' + log_name
components = ['EPE', 'EPN', 'EPZ']  # 根据你的 SAC 数据调整
sac_events = [f for f in os.listdir(h5_file) if f.endswith('.h5')]
sac_events.sort(key=extract_date_safe)
for threshold in threshlolds:
	for weight in weights:
		result_file = 'matched_filter_results_threshold' + threshold + '_weight' + str(weight[1]) + '.h5'
		result_dict = utils.load_result(result_file, path=result_dir)
		templates = list(result_dict.keys())
		save_dir = './analysis_plots/' + log_name + 'matched_filter_results_threshold'+threshold+'_weight' + str(weight[1]) + '/'
		# 提取匹配事件波形

		# 对每个模板绘制波形图
		for template_idx,template_event in enumerate(templates):
			#if result_dict[template_event]['matched_count'] > 150000:
			#	print(f"{template_name} too much event. Skipping")
			#	continue
			template_name = template_event.split('.')[0]
			if os.path.exists(save_dir + template_name + '_' + components[weight[0]]+'.png'):
				print(f"File {save_dir + template_name + '_' + components[weight[0]]} already exists. Skipping.")  
			else:
				print(f"Processing template: {template_name}")
				detection = extract_matched_waveforms_from_daily(
					template_event, n_best, result_dict, sac_events,data_dir = h5_file
				)
				if len(detection['waveforms']) == 0:
					print(f"No detections found for {template_name}, skipping.")
					continue

				template = utils.load_template(template_event, path='./templates/')
				# 绘图
				plot_n_detections(
					detection=detection,
					n_best= n_best,
					template=template,
					components=components,
					weight = weight[0],
					station_idx= 0,
					save_pos = save_dir + template_name + '_' + components[weight[0]]
				)

File ./analysis_plots/814-914/matched_filter_results_threshold0.5_weight0.0_0.0_1.0/template_15021000_EPZ already exists. Skipping.
File ./analysis_plots/814-914/matched_filter_results_threshold0.5_weight0.0_0.0_1.0/template_15029300_EPZ already exists. Skipping.
Processing template: template_15390050


: 